In [53]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [54]:
# 1. INIT SPARK
spark_session = SparkSession.builder \
    .appName("SellerRiskAnalysis") \
     .config("spark.jars", "/Users/as-mac-1261/Downloads/mysql-connector-j-9.6.0/mysql-connector-j-9.6.0.jar") \
    .getOrCreate()

spark.conf.set("spark.sql.ansi.enabled", "false")
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")


In [56]:
# READ FILES
orders_df = spark_session.read.csv("/Users/as-mac-1261/sample/sample/data/orders_fact.csv", header=True, inferSchema=True)
returns_df = spark_session.read.csv("/Users/as-mac-1261/sample/sample/data/returns_events.csv", header=True, inferSchema=True)

orders_df.show()
returns_df.show()

+--------+-------------+---------+----------+----------------+------------+---------------+---------------+------------+------------+----------------+----------+-------------+
|order_id|order_item_id|seller_id|product_id|product_category|order_amount|discount_amount|net_paid_amount|order_status|payment_type|fulfillment_type|order_date|delivery_date|
+--------+-------------+---------+----------+----------------+------------+---------------+---------------+------------+------------+----------------+----------+-------------+
| ORD1001|       IT1001|    S-101|      P501|     Electronics|        2499|            300|           2199|   Completed|        Card|     Marketplace|2026-03-10|   2026-03-13|
| ORD1002|       IT1002|    S-205|      P880|            Home|        1599|            100|           1499|   Completed|         UPI|       Warehouse|2026-03-11|   2026-03-14|
| ORD1003|       IT1003|    S-101|      P502|     Electronics|        1999|            200|           1799|   Completed|

In [57]:
# STANDARDIZE SELLER ID
orders_df = orders_df.withColumn("seller_key", upper(col("seller_id")))
returns_df = returns_df.withColumn("seller_key", upper(col("seller_id")))
orders_df.show()
returns_df.show()



+--------+-------------+---------+----------+----------------+------------+---------------+---------------+------------+------------+----------------+----------+-------------+----------+
|order_id|order_item_id|seller_id|product_id|product_category|order_amount|discount_amount|net_paid_amount|order_status|payment_type|fulfillment_type|order_date|delivery_date|seller_key|
+--------+-------------+---------+----------+----------------+------------+---------------+---------------+------------+------------+----------------+----------+-------------+----------+
| ORD1001|       IT1001|    S-101|      P501|     Electronics|        2499|            300|           2199|   Completed|        Card|     Marketplace|2026-03-10|   2026-03-13|     S-101|
| ORD1002|       IT1002|    S-205|      P880|            Home|        1599|            100|           1499|   Completed|         UPI|       Warehouse|2026-03-11|   2026-03-14|     S-205|
| ORD1003|       IT1003|    S-101|      P502|     Electronics|   

In [58]:
# CONVERT DATE COLUMNS  

orders_df = orders_df.withColumn("delivery_date", to_date(col("delivery_date")))
returns_df = returns_df.withColumn("return_date", to_date(col("return_date")))

orders_df.show()
returns_df.show()


+--------+-------------+---------+----------+----------------+------------+---------------+---------------+------------+------------+----------------+----------+-------------+----------+
|order_id|order_item_id|seller_id|product_id|product_category|order_amount|discount_amount|net_paid_amount|order_status|payment_type|fulfillment_type|order_date|delivery_date|seller_key|
+--------+-------------+---------+----------+----------------+------------+---------------+---------------+------------+------------+----------------+----------+-------------+----------+
| ORD1001|       IT1001|    S-101|      P501|     Electronics|        2499|            300|           2199|   Completed|        Card|     Marketplace|2026-03-10|   2026-03-13|     S-101|
| ORD1002|       IT1002|    S-205|      P880|            Home|        1599|            100|           1499|   Completed|         UPI|       Warehouse|2026-03-11|   2026-03-14|     S-205|
| ORD1003|       IT1003|    S-101|      P502|     Electronics|   

In [60]:
#  FILTER COMPLETED ORDERS

orders_df = orders_df.filter(col("order_status") == "Completed")

returns_df = returns_df.fillna({
    "refund_amount": 0.0
})

In [62]:
# 5. DEDUPLICATE RETURNS
returns_df = returns_df.withColumn(
    "snapshot_ts", to_timestamp(col("snapshot_ts"))
)

window_spec = Window.partitionBy("order_item_id") \
                    .orderBy(col("snapshot_ts").desc())

returns_df = returns_df.withColumn("row_num", row_number().over(window_spec)) \
                       .filter(col("row_num") == 1) \
                       .drop("row_num")

returns_df.show()


+---------+--------+-------------+---------+-------------+-------------+-------------+-------------+-----------+-------------+-------------------+----------+
|return_id|order_id|order_item_id|seller_id|return_reason|refund_amount|return_status|pickup_status|return_date|source_system|        snapshot_ts|seller_key|
+---------+--------+-------------+---------+-------------+-------------+-------------+-------------+-----------+-------------+-------------------+----------+
|  RET9001| ORD1001|       IT1001|    S-101| Damaged item|         2199|     Approved|    Completed| 2026-03-18|      MarketA|2026-03-19 02:10:00|     S-101|
|  RET9002| ORD1002|       IT1002|    S-205|Late Delivery|         1499|     Approved|    Completed| 2026-03-20|      MarketA|2026-03-20 03:00:00|     S-205|
|  RET9003| ORD1003|       IT1003|    S-101|    Defective|         1799|     Approved|    Completed| 2026-03-21|      MarketA|2026-03-21 02:00:00|     S-101|
|  RET9004| ORD1004|       IT1004|    S-202|   Wrong

In [63]:

returns_df = returns_df.withColumnRenamed("seller_key", "seller_key_ret")

#Join 

merged_df = orders_df.join(returns_df, ["order_id", "order_item_id"], "left")

merged_df.show()


+--------+-------------+---------+----------+----------------+------------+---------------+---------------+------------+------------+----------------+----------+-------------+----------+---------+---------+-------------+-------------+-------------+-------------+-----------+-------------+-------------------+--------------+
|order_id|order_item_id|seller_id|product_id|product_category|order_amount|discount_amount|net_paid_amount|order_status|payment_type|fulfillment_type|order_date|delivery_date|seller_key|return_id|seller_id|return_reason|refund_amount|return_status|pickup_status|return_date|source_system|        snapshot_ts|seller_key_ret|
+--------+-------------+---------+----------+----------------+------------+---------------+---------------+------------+------------+----------------+----------+-------------+----------+---------+---------+-------------+-------------+-------------+-------------+-----------+-------------+-------------------+--------------+
| ORD1001|       IT1001|    

In [ ]:

#  FEATURES
merged_df = merged_df.withColumn(
    "return_days",
    datediff(col("return_date"), col("delivery_date"))
)

merged_df = merged_df.withColumn(
    "refund_ratio",
    when(col("net_paid_amount") != 0,
         col("refund_amount") / col("net_paid_amount")
    ).otherwise(0)
)

merged_df = merged_df.withColumn(
    "damage_flag",
    when(lower(col("return_reason")).contains("damaged"), 1).otherwise(0)
)

merged_df = merged_df.withColumn(
    "late_flag",
    when(col("return_days") > 7, 1).otherwise(0)
)

merged_df = merged_df.withColumn(
    "high_refund_flag",
    when(col("refund_ratio") > 0.8, 1).otherwise(0)
)

merged_df.show()

+--------+-------------+---------+----------+----------------+------------+---------------+---------------+------------+------------+----------------+----------+-------------+----------+---------+---------+-------------+-------------+-------------+-------------+-----------+-------------+-------------------+--------------+-----------+------------+-----------+---------+----------------+
|order_id|order_item_id|seller_id|product_id|product_category|order_amount|discount_amount|net_paid_amount|order_status|payment_type|fulfillment_type|order_date|delivery_date|seller_key|return_id|seller_id|return_reason|refund_amount|return_status|pickup_status|return_date|source_system|        snapshot_ts|seller_key_ret|return_days|refund_ratio|damage_flag|late_flag|high_refund_flag|
+--------+-------------+---------+----------+----------------+------------+---------------+---------------+------------+------------+----------------+----------+-------------+----------+---------+---------+-------------+----

In [ ]:

# AGGREGATION

result_df = merged_df.groupBy("seller_key") \
    .agg(
        count("order_id").alias("total_orders"),
        count("return_id").alias("total_returns"),
        sum("refund_amount").alias("total_refund"),
        avg("refund_amount").alias("avg_refund"),
        avg("damage_flag").alias("damage_ratio"),
        avg("late_flag").alias("late_ratio"),
        sum("high_refund_flag").alias("high_refund_cases")
    )

result_df = result_df.withColumn(
    "return_rate",
    when(col("total_orders") != 0,
         col("total_returns") / col("total_orders")
    ).otherwise(0)
)

result_df.show()

+----------+------------+-------------+------------+-----------------+-------------------+----------+-----------------+-----------+
|seller_key|total_orders|total_returns|total_refund|       avg_refund|       damage_ratio|late_ratio|high_refund_cases|return_rate|
+----------+------------+-------------+------------+-----------------+-------------------+----------+-----------------+-----------+
|     S-303|           5|            5|        4545|            909.0|                0.6|       0.0|                5|        1.0|
|     S-505|           4|            4|        5046|           1261.5|               0.25|       0.0|                4|        1.0|
|     S-101|           7|            7|       16893|2413.285714285714|0.42857142857142855|       0.0|                7|        1.0|
|     S-202|           5|            5|        5495|           1099.0|                0.2|       0.0|                5|        1.0|
|     S-404|           4|            4|        2396|            599.0|      

In [66]:

# RISK SCORE
result_df = result_df.withColumn(
    "risk_score",
    col("return_rate") * 0.5 +
    col("damage_ratio") * 0.3 +
    col("late_ratio") * 0.2
)

final_output = result_df.orderBy(col("risk_score").desc())

final_output.show()

+----------+------------+-------------+------------+-----------------+-------------------+----------+-----------------+-----------+------------------+
|seller_key|total_orders|total_returns|total_refund|       avg_refund|       damage_ratio|late_ratio|high_refund_cases|return_rate|        risk_score|
+----------+------------+-------------+------------+-----------------+-------------------+----------+-----------------+-----------+------------------+
|     S-303|           5|            5|        4545|            909.0|                0.6|       0.0|                5|        1.0|0.6799999999999999|
|     S-404|           4|            4|        2396|            599.0|                0.5|       0.0|                4|        1.0|              0.65|
|     S-101|           7|            7|       16893|2413.285714285714|0.42857142857142855|       0.0|                7|        1.0|0.6285714285714286|
|     S-205|           5|            5|        8495|           1699.0|                0.4|    

In [67]:

# 10. WRITE TO MYSQL
mysql_url = "jdbc:mysql://localhost:3306/assignment"

properties = {
    "user": "root",
    "password": "Jeevan@123",
    "driver": "com.mysql.cj.jdbc.Driver"
}

final_df.write.jdbc(url=mysql_url, table="seller_risk", mode="overwrite", properties=properties)
final_df.show()
print("Database conencted successfully")

+-------------+----------------------+--------------------+-------------------+-----------------+-------------------+-----------------+----------------------+-----------+------------------+
|seller_id_std|total_completed_orders|total_returned_items|total_refund_amount|avg_refund_amount|damaged_claim_ratio|late_return_ratio|high_refund_case_count|return_rate| seller_risk_score|
+-------------+----------------------+--------------------+-------------------+-----------------+-------------------+-----------------+----------------------+-----------+------------------+
|        S-303|                     5|                   5|               4545|            909.0|                0.6|              0.0|                     5|        1.0|0.6799999999999999|
|        S-404|                     4|                   4|               2396|            599.0|                0.5|              0.0|                     4|        1.0|              0.65|
|        S-101|                     7|            